In [30]:
import tensorflow as tf
import numpy as np
print(tf.__version__)

2.5.0


In [31]:
# create a tensorflow tensor object, 1D tensor
a = tf.Variable([1,2,3,4,5])
a

<tf.Variable 'Variable:0' shape=(5,) dtype=int32, numpy=array([1, 2, 3, 4, 5], dtype=int32)>

In [32]:
# 2d tensor
b = tf.Variable([[1,2,3], [4,5,6]])
b

<tf.Variable 'Variable:0' shape=(2, 3) dtype=int32, numpy=
array([[1, 2, 3],
       [4, 5, 6]], dtype=int32)>

In [33]:
# 3d tensor
c = tf.Variable([[[1,2,3], [4,5,6]], [[1,2,3], [4,5,6]]])
c

<tf.Variable 'Variable:0' shape=(2, 2, 3) dtype=int32, numpy=
array([[[1, 2, 3],
        [4, 5, 6]],

       [[1, 2, 3],
        [4, 5, 6]]], dtype=int32)>

In [34]:
# tensor indexing
a = tf.Variable([[1,2,3], [4,5,6], [7,8,9]])
print(a)
print(a[1,1])
print(a[1,:])
print(a[1:3,1:3])
print(a[1][2])

<tf.Variable 'Variable:0' shape=(3, 3) dtype=int32, numpy=
array([[1, 2, 3],
       [4, 5, 6],
       [7, 8, 9]], dtype=int32)>
tf.Tensor(5, shape=(), dtype=int32)
tf.Tensor([4 5 6], shape=(3,), dtype=int32)
tf.Tensor(
[[5 6]
 [8 9]], shape=(2, 2), dtype=int32)
tf.Tensor(6, shape=(), dtype=int32)


In [35]:
# slicing 2D arrays
a = tf.Variable([[1,2,3,4,5], [6,7,8,9,10]])
print(a[0:2, -3])
print(a[0:2, 1:4])

tf.Tensor([3 8], shape=(2,), dtype=int32)
tf.Tensor(
[[2 3 4]
 [7 8 9]], shape=(2, 3), dtype=int32)


# Vectorized operations
- Math operation: +, -, *, /, **,
- Logical operations: And:&, Or:|, Not:~. Only valid for boolean. If use it on integers, then it's bitwise operation .
- Comparison operations: >, >=, <=, <, ==, !=.


In [36]:
# vectorized operations
a = tf.Variable([1,2,3])
b = tf.Variable([4,5,6])
print("a + b = {}".format(a+b))
print("a * b = {}".format(a*b))
print("a * 3 = {}".format(a*3))
print("a & 3 = {}".format(a&b))
print("a > 3 = {}".format(a>b))

a + b = [5 7 9]
a * b = [ 4 10 18]
a * 3 = [3 6 9]
a & 3 = [0 0 2]
a > 3 = [False False False]


# Shape of the tensor

In [37]:
a = tf.Variable([[[0, 1, 2, 3], [4,5,6,7]], [[0,1,2,3], [4,5,6,7]], [[0,1,2,3], [4,5,6,7]]])
print(a.shape)
print(tf.size(a))

(3, 2, 4)
tf.Tensor(24, shape=(), dtype=int32)


# Transposing and reshaping index

In [38]:
data = tf.constant([[1,2], [3,4], [5,6]])
print(tf.transpose(data))

tf.Tensor(
[[1 3 5]
 [2 4 6]], shape=(2, 3), dtype=int32)


# Convolutional Neural Network Example

# Build from scratch

MNIST Dataset : http://yann.lecun.com/exdb/mnist/

In [39]:
from __future__ import absolute_import, division, print_function

import tensorflow as tf
import numpy as np

In [40]:
# MNIST dataset parameters.
num_classes = 10 # total classes (0-9 digits).

# Training parameters.
learning_rate = 0.001
training_steps = 200
batch_size = 128
display_step = 10

# Network parameters.
conv1_filters = 32 # number of filters for 1st conv layer.
conv2_filters = 64 # number of filters for 2nd conv layer.
fc1_units = 1024 # number of neurons for 1st fully-connected layer.

In [59]:
# Prepare MNIST data.
from tensorflow.keras.datasets import mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()
# Convert to float32.
x_train, x_test = np.array(x_train, np.float32), np.array(x_test, np.float32)
# Normalize images value from [0, 255] to [0, 1].
x_train, x_test = x_train / 255., x_test / 255.
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(60000, 28, 28)
(60000,)
(10000, 28, 28)
(10000,)


In [42]:
# Use tf.data API to shuffle and batch data.
train_data = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_data = train_data.repeat().shuffle(5000).batch(batch_size).prefetch(1)

In [43]:
# Create some wrappers for simplicity.
def conv2d(x, W, b, strides=1):
    # Conv2D wrapper, with bias and relu activation.
    x = tf.nn.conv2d(x, W, strides=[1, strides, strides, 1], padding='SAME')
    x = tf.nn.bias_add(x, b)
    return tf.nn.relu(x)

def maxpool2d(x, k=2):
    # MaxPool2D wrapper.
    return tf.nn.max_pool(x, ksize=[1, k, k, 1], strides=[1, k, k, 1], padding='SAME')

In [44]:
# Store layers weight & bias

# A random value generator to initialize weights.
random_normal = tf.initializers.RandomNormal()

weights = {
    # Conv Layer 1: 5x5 conv, 1 input, 32 filters (MNIST has 1 color channel only).
    'wc1': tf.Variable(random_normal([5, 5, 1, conv1_filters])),
    # Conv Layer 2: 5x5 conv, 32 inputs, 64 filters.
    'wc2': tf.Variable(random_normal([5, 5, conv1_filters, conv2_filters])),
    # FC Layer 1: 7*7*64 inputs, 1024 units.
    'wd1': tf.Variable(random_normal([7*7*64, fc1_units])),
    # FC Out Layer: 1024 inputs, 10 units (total number of classes)
    'out': tf.Variable(random_normal([fc1_units, num_classes]))
}

biases = {
    'bc1': tf.Variable(tf.zeros([conv1_filters])),
    'bc2': tf.Variable(tf.zeros([conv2_filters])),
    'bd1': tf.Variable(tf.zeros([fc1_units])),
    'out': tf.Variable(tf.zeros([num_classes]))
}

In [45]:
# Create model
def conv_net(x):
    
    # Input shape: [-1, 28, 28, 1]. A batch of 28x28x1 (grayscale) images.
    x = tf.reshape(x, [-1, 28, 28, 1])

    # Convolution Layer. Output shape: [-1, 28, 28, 32].
    conv1 = conv2d(x, weights['wc1'], biases['bc1'])
    
    # Max Pooling (down-sampling). Output shape: [-1, 14, 14, 32].
    conv1 = maxpool2d(conv1, k=2)

    # Convolution Layer. Output shape: [-1, 14, 14, 64].
    conv2 = conv2d(conv1, weights['wc2'], biases['bc2'])
    
    # Max Pooling (down-sampling). Output shape: [-1, 7, 7, 64].
    conv2 = maxpool2d(conv2, k=2)

    # Reshape conv2 output to fit fully connected layer input, Output shape: [-1, 7*7*64].
    fc1 = tf.reshape(conv2, [-1, weights['wd1'].get_shape().as_list()[0]])
    
    # Fully connected layer, Output shape: [-1, 1024].
    fc1 = tf.add(tf.matmul(fc1, weights['wd1']), biases['bd1'])
    # Apply ReLU to fc1 output for non-linearity.
    fc1 = tf.nn.relu(fc1)

    # Fully connected layer, Output shape: [-1, 10].
    out = tf.add(tf.matmul(fc1, weights['out']), biases['out'])
    # Apply softmax to normalize the logits to a probability distribution.
    return tf.nn.softmax(out)

In [46]:
# Cross-Entropy loss function.
def cross_entropy(y_pred, y_true):
    # Encode label to a one hot vector.
    y_true = tf.one_hot(y_true, depth=num_classes)
    # Clip prediction values to avoid log(0) error.
    y_pred = tf.clip_by_value(y_pred, 1e-9, 1.)
    # Compute cross-entropy.
    return tf.reduce_mean(-tf.reduce_sum(y_true * tf.math.log(y_pred)))

# Accuracy metric.
def accuracy(y_pred, y_true):
    # Predicted class is the index of highest score in prediction vector (i.e. argmax).
    correct_prediction = tf.equal(tf.argmax(y_pred, 1), tf.cast(y_true, tf.int64))
    return tf.reduce_mean(tf.cast(correct_prediction, tf.float32), axis=-1)

# ADAM optimizer.
optimizer = tf.optimizers.Adam(learning_rate)

In [47]:
# Optimization process. 
def run_optimization(x, y):
    # Wrap computation inside a GradientTape for automatic differentiation.
    with tf.GradientTape() as g:
        pred = conv_net(x)
        loss = cross_entropy(pred, y)
        
    # Variables to update, i.e. trainable variables.
    trainable_variables = list(weights.values()) + list(biases.values())

    # Compute gradients.
    gradients = g.gradient(loss, trainable_variables)
    
    # Update W and b following gradients.
    optimizer.apply_gradients(zip(gradients, trainable_variables))

In [48]:
# Run training for the given number of steps.
for step, (batch_x, batch_y) in enumerate(train_data.take(training_steps), 1):
    # Run the optimization to update W and b values.
    run_optimization(batch_x, batch_y)
    
    if step % display_step == 0:
        pred = conv_net(batch_x)
        loss = cross_entropy(pred, batch_y)
        acc = accuracy(pred, batch_y)
        print("step: %i, loss: %f, accuracy: %f" % (step, loss, acc))

step: 10, loss: 54.793621, accuracy: 0.921875
step: 20, loss: 44.381191, accuracy: 0.875000
step: 30, loss: 38.055222, accuracy: 0.945312
step: 40, loss: 27.181644, accuracy: 0.945312
step: 50, loss: 19.609373, accuracy: 0.960938
step: 60, loss: 27.537004, accuracy: 0.945312
step: 70, loss: 15.918192, accuracy: 0.960938
step: 80, loss: 13.479110, accuracy: 0.960938
step: 90, loss: 18.232195, accuracy: 0.960938
step: 100, loss: 13.551282, accuracy: 0.968750
step: 110, loss: 14.267879, accuracy: 0.960938
step: 120, loss: 17.265507, accuracy: 0.953125
step: 130, loss: 6.305201, accuracy: 0.984375
step: 140, loss: 4.396489, accuracy: 0.992188
step: 150, loss: 10.964341, accuracy: 0.976562
step: 160, loss: 4.182031, accuracy: 1.000000
step: 170, loss: 8.233334, accuracy: 0.976562
step: 180, loss: 6.904018, accuracy: 0.968750
step: 190, loss: 5.251275, accuracy: 0.992188
step: 200, loss: 5.060575, accuracy: 0.992188


In [49]:
# Test model on validation set.
pred = conv_net(x_test)
print("Test Accuracy: %f" % accuracy(pred, y_test))

Test Accuracy: 0.976500


# Use highlevel API

In [50]:
from __future__ import absolute_import, division, print_function

import tensorflow as tf
from tensorflow.keras import Model, layers
import numpy as np

In [51]:
# MNIST dataset parameters.
num_classes = 10 # total classes (0-9 digits).

# Training parameters.
learning_rate = 0.001
training_steps = 200
batch_size = 128
display_step = 10

# Network parameters.
conv1_filters = 32 # number of filters for 1st conv layer.
conv2_filters = 64 # number of filters for 2nd conv layer.
fc1_units = 1024 # number of neurons for 1st fully-connected layer.

In [52]:
# Prepare MNIST data.
from tensorflow.keras.datasets import mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()
# Convert to float32.
x_train, x_test = np.array(x_train, np.float32), np.array(x_test, np.float32)
# Normalize images value from [0, 255] to [0, 1].
x_train, x_test = x_train / 255., x_test / 255.

In [53]:
# Use tf.data API to shuffle and batch data.
train_data = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_data = train_data.repeat().shuffle(5000).batch(batch_size).prefetch(1)

In [54]:
# Create TF Model.
class ConvNet(Model):
    # Set layers.
    def __init__(self):
        super(ConvNet, self).__init__()
        # Convolution Layer with 32 filters and a kernel size of 5.
        self.conv1 = layers.Conv2D(32, kernel_size=5, activation=tf.nn.relu)
        # Max Pooling (down-sampling) with kernel size of 2 and strides of 2. 
        self.maxpool1 = layers.MaxPool2D(2, strides=2)

        # Convolution Layer with 64 filters and a kernel size of 3.
        self.conv2 = layers.Conv2D(64, kernel_size=3, activation=tf.nn.relu)
        # Max Pooling (down-sampling) with kernel size of 2 and strides of 2. 
        self.maxpool2 = layers.MaxPool2D(2, strides=2)

        # Flatten the data to a 1-D vector for the fully connected layer.
        self.flatten = layers.Flatten()

        # Fully connected layer.
        self.fc1 = layers.Dense(1024)
        # Apply Dropout (if is_training is False, dropout is not applied).
        self.dropout = layers.Dropout(rate=0.5)

        # Output layer, class prediction.
        self.out = layers.Dense(num_classes)

    # Set forward pass.
    def call(self, x, is_training=False):
        x = tf.reshape(x, [-1, 28, 28, 1])
        x = self.conv1(x)
        x = self.maxpool1(x)
        x = self.conv2(x)
        x = self.maxpool2(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.dropout(x, training=is_training)
        x = self.out(x)
        if not is_training:
            # tf cross entropy expect logits without softmax, so only
            # apply softmax when not training.
            x = tf.nn.softmax(x)
        return x

# Build neural network model.
conv_net = ConvNet()

In [55]:
# Cross-Entropy Loss.
# Note that this will apply 'softmax' to the logits.
def cross_entropy_loss(x, y):
    # Convert labels to int 64 for tf cross-entropy function.
    y = tf.cast(y, tf.int64)
    # Apply softmax to logits and compute cross-entropy.
    loss = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y, logits=x)
    # Average loss across the batch.
    return tf.reduce_mean(loss)

# Accuracy metric.
def accuracy(y_pred, y_true):
    # Predicted class is the index of highest score in prediction vector (i.e. argmax).
    correct_prediction = tf.equal(tf.argmax(y_pred, 1), tf.cast(y_true, tf.int64))
    return tf.reduce_mean(tf.cast(correct_prediction, tf.float32), axis=-1)

# Stochastic gradient descent optimizer.
optimizer = tf.optimizers.Adam(learning_rate)

In [56]:
# Optimization process. 
def run_optimization(x, y):
    # Wrap computation inside a GradientTape for automatic differentiation.
    with tf.GradientTape() as g:
        # Forward pass.
        pred = conv_net(x, is_training=True)
        # Compute loss.
        loss = cross_entropy_loss(pred, y)
        
    # Variables to update, i.e. trainable variables.
    trainable_variables = conv_net.trainable_variables

    # Compute gradients.
    gradients = g.gradient(loss, trainable_variables)
    
    # Update W and b following gradients.
    optimizer.apply_gradients(zip(gradients, trainable_variables))

In [57]:
# Run training for the given number of steps.
for step, (batch_x, batch_y) in enumerate(train_data.take(training_steps), 1):
    # Run the optimization to update W and b values.
    run_optimization(batch_x, batch_y)
    
    if step % display_step == 0:
        pred = conv_net(batch_x)
        loss = cross_entropy_loss(pred, batch_y)
        acc = accuracy(pred, batch_y)
        print("step: %i, loss: %f, accuracy: %f" % (step, loss, acc))

step: 10, loss: 1.891207, accuracy: 0.750000
step: 20, loss: 1.636364, accuracy: 0.906250
step: 30, loss: 1.582116, accuracy: 0.937500
step: 40, loss: 1.562276, accuracy: 0.937500
step: 50, loss: 1.560085, accuracy: 0.937500
step: 60, loss: 1.536458, accuracy: 0.937500
step: 70, loss: 1.519346, accuracy: 0.968750
step: 80, loss: 1.543672, accuracy: 0.945312
step: 90, loss: 1.523446, accuracy: 0.953125
step: 100, loss: 1.513796, accuracy: 0.968750
step: 110, loss: 1.522298, accuracy: 0.960938
step: 120, loss: 1.515992, accuracy: 0.976562
step: 130, loss: 1.504946, accuracy: 0.976562
step: 140, loss: 1.518155, accuracy: 0.953125
step: 150, loss: 1.514156, accuracy: 0.960938
step: 160, loss: 1.523908, accuracy: 0.953125
step: 170, loss: 1.497031, accuracy: 0.976562
step: 180, loss: 1.507106, accuracy: 0.984375
step: 190, loss: 1.513000, accuracy: 0.968750
step: 200, loss: 1.489849, accuracy: 1.000000


In [58]:
# Test model on validation set.
pred = conv_net(x_test)
print("Test Accuracy: %f" % accuracy(pred, y_test))

Test Accuracy: 0.976600
